In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [2]:
spark = SparkSession.builder \
    .appName("Olist_ETL") \
    .master("yarn") \
    .config("spark.submit.deployMode", "client") \
    .enableHiveSupport() \
    .getOrCreate()


## Create gold Database / schema

In [8]:
spark.sql("create database if not exists gold;")

2026-04-26 04:19:07,532 ERROR metastore.RetryingHMSHandler: AlreadyExistsException(message:Database gold already exists)
	at org.apache.hadoop.hive.metastore.HiveMetaStore$HMSHandler.create_database(HiveMetaStore.java:925)
	at sun.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at sun.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at sun.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.lang.reflect.Method.invoke(Method.java:498)
	at org.apache.hadoop.hive.metastore.RetryingHMSHandler.invokeInternal(RetryingHMSHandler.java:148)
	at org.apache.hadoop.hive.metastore.RetryingHMSHandler.invoke(RetryingHMSHandler.java:107)
	at com.sun.proxy.$Proxy27.create_database(Unknown Source)
	at org.apache.hadoop.hive.metastore.HiveMetaStoreClient.createDatabase(HiveMetaStoreClient.java:727)
	at sun.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at sun.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImp

DataFrame[]

## **Sales & Revenue**


### What is the total revenue per year?


In [13]:
spark.sql(''' 
CREATE table if NOT EXISTS gold.revenue_per_year  as
SELECT 
  year(o.order_purchase_timestamp) as year
  ,round(sum(oi.price+oi.freight_value),2) as Total_Revenue  
from silver.order_items  oi join silver.orders  o
  on o.order_id = oi.order_id
WHERE o.order_status = 'delivered'    
GROUP BY year(o.order_purchase_timestamp)
ORDER BY year(o.order_purchase_timestamp)
;
'''
)

2026-04-26 04:50:21,856 WARN analysis.ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
2026-04-26 04:50:21,971 WARN metastore.HiveMetaStore: Location: hdfs://localhost:9000/user/hive/warehouse/gold.db/revenue_per_year specified for non-external table:revenue_per_year


DataFrame[]

### Which product categories drive the most sales?


In [10]:
spark.sql('''
CREATE TABLE if NOT EXISTS gold.revenue_per_category  
SELECT 
  p.product_category_name as category
  ,round(sum(oi.price+oi.freight_value),2) as Total_Revenue  
from silver.order_items  oi join silver.products  p
  on oi.product_id = p.product_id
  JOIN silver.orders o 
  on oi.order_id=o.order_id
WHERE o.order_status = 'delivered'    
GROUP BY p.product_category_name
ORDER BY p.product_category_name
;

''')

2026-04-26 04:48:23,912 WARN analysis.ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
2026-04-26 04:48:25,154 WARN metastore.HiveMetaStore: Location: hdfs://localhost:9000/user/hive/warehouse/gold.db/revenue_per_category specified for non-external table:revenue_per_category


DataFrame[]

## Customer Behaviour 

### Who are the top customers by spend ? (Top 10)

In [14]:
spark.sql('''
CREATE TABLE IF NOT EXISTS gold.top_performing_customers
SELECT 
  c.customer_id as customer_id
  ,round(sum(oi.price+oi.freight_value),2) as Total_Revenue  
from silver.orders o join silver.customers c
  on o.customer_id = c.customer_id
  JOIN silver.order_items oi 
  on oi.order_id=o.order_id
WHERE o.order_status = 'delivered'    
GROUP BY c.customer_id
ORDER BY Total_Revenue desc
LIMIT 10
;

''')

2026-04-26 05:00:16,692 WARN analysis.ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
2026-04-26 05:00:17,694 WARN metastore.HiveMetaStore: Location: hdfs://localhost:9000/user/hive/warehouse/gold.db/top_performing_customers specified for non-external table:top_performing_customers


DataFrame[]

## Order Operations 

### What is the order status breakdown?

In [18]:
spark.sql('''
CREATE TABLE IF NOT EXISTS gold.order_breakdown
SELECT
    order_status,
    COUNT(*)                                            AS total_orders,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2)  AS pct_of_total
 
FROM silver.orders
 
GROUP BY order_status
ORDER BY total_orders DESC;
 
''')

2026-04-26 05:14:15,954 WARN analysis.ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
2026-04-26 05:14:15,998 WARN window.WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
2026-04-26 05:14:16,131 WARN metastore.HiveMetaStore: Location: hdfs://localhost:9000/user/hive/warehouse/gold.db/order_breakdown specified for non-external table:order_breakdown


DataFrame[]

### What is the average delivery time?

In [22]:
spark.sql('''
CREATE TABLE IF NOT EXISTS gold.avg_delivry_days
select avg(delivery_days) as avg_delivry_days 
from silver.orders
''')

2026-04-26 05:17:00,376 WARN analysis.ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.


DataFrame[]

## Customer Satisfaction

### What is the average review score by category? (top 10)

In [40]:
spark.sql('''
CREATE TABLE IF NOT EXISTS gold.avg_rev_score_per_category
SELECT 
    p.product_category_name ,
    round(avg(or.review_score),2) as avg_review_score
FROM 
    silver.products p join silver.order_items oi
    on p.product_id = oi.product_id
    join silver.order_reviews or
    on oi.order_id = or.order_id
GROUP By p.product_category_name    
limit 10;

''')

2026-04-26 05:38:01,131 WARN analysis.ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
2026-04-26 05:38:01,217 WARN metastore.HiveMetaStore: Location: hdfs://localhost:9000/user/hive/warehouse/gold.db/avg_rev_score_per_category specified for non-external table:avg_rev_score_per_category


DataFrame[]